# CarbonGlobe Data Dictionary

This dictionary combines the dataset description available to the team with array shapes observed during exploration. Items marked **reported; verify storage/mapping** come from the high-level metadata description and have not yet been matched to a specific file or array column.

## Dataset and observation structure

| Item | Description | Status / notes |
|---|---|---|
| Dataset purpose | ML-ready dataset for forecasting the forest carbon cycle, based on Ecosystem Demography (ED) v3.0 simulations. | Reported in the dataset overview. |
| Global sites | 54,152 land sites on a 0.5° grid. | Global input shape begins with 54,152 sites. |
| Input array | `glob_X_fea.npy`: `(site, year, month, feature) = (54152, 40, 12, 136)`. | One selected `X[site, year, month, :]` is one site's one-month observation with 136 feature values. |
| Target array | `glob_Y{age}.npy`: `(site, year, month, target) = (54152, 41, 12, 7)` for a seed-tree age. | Year index 0 is described as the initial state; later positions are forecast states. Verify exact time alignment. |
| Forecast horizon | 40 years, run autoregressively. | Reported in the task description; 41 target-year positions include the initial state. |
| Seed-tree ages | 15 ages: 1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500 years. | Reported; outputs are age-specific. |
| Time coverage | Monthly; years 1981–2021 are reported in the metadata description. | **Needs reconciliation:** the arrays have 40 input-year positions and 41 target-year positions including initialization. Confirm the date/index convention in the official documentation. |

The raw arrays are multidimensional, not ordinary row-and-column tables. If reshaped, define the row meaning explicitly. For example, a flattened input row could represent one site-month, with 136 feature columns; a supervised training row may additionally include the prior target state, seed age, and the next target to predict.

## Input feature groups

| Group | Meaning | Status / notes |
|---|---|---|
| Forcing drivers | Environmental drivers that force the ecosystem simulation. | Group reported; exact variables, channel indices, and units are not yet mapped. |
| CO₂ | Carbon-dioxide input(s). | Group reported; exact variable names, channel indices, and units are not yet mapped. |
| Soil | Soil input properties or conditions. | Group reported; exact variables, channel indices, and units are not yet mapped. |
| Vegetation indices | Vegetation-related index inputs. | Group reported; exact indices, channel mapping, and units are not yet mapped. |

There are 136 input channels in total. The group descriptions do **not** identify which channel number contains which variable. Until the authoritative feature list/order is found, refer to channels as `feature_0` through `feature_135`; do not infer names or units from sample values.

## Target variables

The seven target channels, in the supplied order, are:

| Target index | Name | Meaning |
|---:|---|---|
| 0 | `height` | Canopy height. |
| 1 | `agb` | Above-ground biomass. |
| 2 | `soil` | Soil carbon. |
| 3 | `lai` | Leaf area index. |
| 4 | `gpp` | Gross primary productivity. |
| 5 | `npp` | Net primary productivity. |
| 6 | `rh` | Heterotrophic respiration. |

Units and any additional model-specific definitions should be copied from the official data documentation; they are not established by the array shape alone.

## Site and other metadata

| Metadata | Meaning | Status / notes |
|---|---|---|
| Geographic coordinates (`lat`, `lon`) | Location of a site. | Reported as metadata attributes; exact file, column names, coordinate convention, and mapping to array site indices are not yet confirmed. |
| Dominant forest type | Dominant forest classification for a site. | Reported as metadata; exact source, categories, and site-index mapping are not yet confirmed. |
| Diverse environmental conditions | Sites span different environmental conditions. | Descriptive dataset characteristic, not necessarily a stored identifier or column. |
| Site index | Position identifying a site in the global arrays. | `train.csv`, `val.csv`, and `test.csv` contain site indices; these indices are not themselves latitude/longitude. |
| Split | Train / validation / test site membership. | Reported counts: 3,035 / 338 / 50,779, partitioning 54,152 global sites. A separate `test8` subset is used by the ML-ready package. |

## Files that support interpretation

| File | Contents |
|---|---|
| `data_stats/data_stats.npz` | `x_mean`, `x_std` (136 values each), and `y_mean`, `y_std` (7 values each), used for normalization. |
| `data_stats/age_triplet.npz` | Age values and age-specific target priors: `age`, `age_mean`, `age_slope`. |
| `data_stats/mask2d.npy` | `(360, 720)` grid mask mapping the 54,152 sites onto the global grid. |
| `train.csv`, `val.csv`, `test.csv` | Site-index splits for global arrays. |
| `data_global/glob_X_fea.npy` | Global input array. |
| `data_global/glob_Y{age}.npy` | Global output array for a seed-tree age. |

## Remaining checks before calling this complete

- Find the official ordered list of all 136 input features, with definitions and units, and map it to the last axis of `glob_X_fea.npy`.
- Locate the coordinate and dominant-forest-type metadata, then verify how its rows map to global site indices.
- Confirm the meaning of year/month indices and reconcile the reported 1981–2021 coverage with the 40 input years and 41 target-year positions.
- Check actual arrays for non-finite values, sentinel missing values, constants, and unexpected ranges. Do not assume that zero means missing.
